In [1]:
from pathlib import Path
import numpy as np
import cv2

from src.cilia_detection.utils import yolobbox2bbox
from tqdm.notebook import tqdm
import skimage

In [2]:
def save_cellprofiler_results(path: Path):
    class_label = 0
    binary_mask = np.load(path)
    label_img = skimage.measure.label(binary_mask, connectivity=2)
    regions = skimage.measure.regionprops(label_img)

    with open((path.parent / path.name.replace("_binary", "")).with_suffix(".txt"), 'w') as f:
        for region in regions:
            ymin, xmin, ymax, xmax = region.bbox
            w, h = xmax - xmin, ymax - ymin
            annotation = f'{class_label} {xmin + w / 2} {ymin + h / 2} {w} {h}\n'
            f.write(annotation)

In [3]:
TYPE = "easy"

In [6]:
# generate bboxes txt files

class_label = 0
cell_profiler_results = Path(f"../../data/cilia_dataset/{TYPE}/cellprofiler_results")

for p in tqdm(cell_profiler_results.glob("*.npy")):
    save_cellprofiler_results(p)

0it [00:00, ?it/s]

In [8]:
# generate images with bboxes (Yolo-format)

for img_p in tqdm((cell_profiler_results.parent / "images").glob("*.tiff")):
    img = cv2.imread(img_p.as_posix())
    with open((cell_profiler_results / img_p.name).with_suffix(".txt")) as file:
        for line in file.readlines():
            yolo_bbox = [float(x) for x in line.strip().split(' ')[1:]]
            bbox = [int(x) for x in yolobbox2bbox(yolo_bbox)]
            cv2.rectangle(img, (bbox[0], bbox[1]), (bbox[2], bbox[3]), (255, 255, 255), thickness=2)

    cv2.imwrite((cell_profiler_results / img_p.name).with_suffix(".png").as_posix(), img)

0it [00:00, ?it/s]